<a href="https://colab.research.google.com/github/yulivvv/llm-agent-with-tools/blob/main/llm-agent-with-tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: Проектирование ИИ-агента на базе LLM

В этом домашнем задании вы пройдете путь от создания базового агента с кастомными инструментами до разработки защищенной мультиагентной системы с человеком в контуре (human-in-the-loop).

**Важное напоминание:** В рамках этого ДЗ вы можете использовать **любые технологии и фреймворки** для реализации задач. Однако мы настоятельно рекомендуем использовать **LangChain** для стандартной части и **LangGraph** для продвинутой - они дают удобные абстракции и хорошо документированы.

**Рекомендация по LLM:** Для отладки агентов со сложной логикой вызова инструментов рекомендуем начинать с больших моделей через [OpenRouter](https://openrouter.ai/) или любой другой сервис (к примеру гигачат, яндекс облако).
---

## Структура ДЗ (100 баллов)

| Часть | Подзадание | Баллы |
|---|---|---|
| Стандартная | 1.1 - 1.3 Реализация 3 инструментов | 20 |
| Стандартная | 1.4 Промпт-инженерия и создание ReAct агента | 10 |
| Стандартная | 1.5 Тестирование базового агента | 10 |
| Стандартная | 1.6 Анализ рисков и идеи по улучшению | 10 |
| Продвинутая | 2.1 Переход на LangGraph | 10 |
| Продвинутая | 2.2 - 2.3 Оркестратор и субагенты | 15 |
| Продвинутая | 2.4 Human-in-the-loop | 10 |
| Продвинутая | 2.5 Финальное тестирование | 5 |
| Продвинутая | 2.6 Анализ рисков и идеи по улучшению | 10 |


---
## Установка зависимостей

Установите необходимые библиотеки. Если вы выбрали инструменты, требующие дополнительных пакетов (например, `yfinance` для курсов валют или `feedparser` для новостей), добавьте их сюда.


In [1]:
!pip install -qU langchain langchain-openai langchain-community langchain-core langgraph datasets matplotlib pandas requests feedparser yfinance

In [42]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

# Берем API-ключ из Google Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

# Адрес OpenRouter
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Инициализация LLM через OpenRouter
llm = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0
)


---
# Часть 1. Стандартная (50 баллов)

В этой части вам нужно:
1. Выбрать и реализовать три инструмента из предложенного списка.
2. Написать системный промпт и создать ReAct агента.
3. Протестировать агента на разных запросах.
4. Проанализировать риски и предложить идеи по улучшению.


### 1.1 - 1.3 Реализация инструментов (20 баллов)

Выберите **три любых инструмента** из списка ниже и реализуйте их:

1. **Поиск по базе знаний** - загрузите датасет `data-silence/rus_news_classifier` с HuggingFace (около 70k коротких русских новостей, поля: `news` - текст, `labels` - категория). Реализуйте поиск по ключевым словам или TF-IDF.
2. **Калькулятор сложных процентов** - функция принимает начальную сумму, годовую ставку (%), срок в годах и частоту капитализации в год.
3. **Построение графиков** - принимает данные (или путь к CSV), строит график через Matplotlib, сохраняет в файл и возвращает путь к нему.
4. **Текущий курс валют** - через публичный API ЦБ РФ (`https://cbr.ru/scripts/XML_daily.asp`, без ключа) или через `yfinance`.
5. **Текущая погода** - через `wttr.in` (без ключа, например: `requests.get("https://wttr.in/Москва?format=j1")`).
6. **Последние новости** - парсинг RSS-ленты любого СМИ через `feedparser` (например, `https://lenta.ru/rss/news`).

**Подсказки по реализации инструментов в LangChain:**
- Используйте декоратор `@tool` из `langchain_core.tools`.
- Пишите подробные docstring - именно по ним LLM понимает, когда и как вызывать инструмент.
- Указывайте типы аргументов (type hints) - это помогает LLM правильно формировать вызов.
- Инструмент должен возвращать строку или что-то, что легко преобразуется в строку.
- Обрабатывайте исключения внутри инструмента и возвращайте понятное сообщение об ошибке.

Пример структуры инструмента:
```python
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Возвращает текущую погоду в указанном городе.
    Используй этот инструмент, когда пользователь спрашивает о погоде.
    Args:
        city: Название города на русском или английском языке.
    """
    try:
        # ваша реализация
        pass
    except Exception as e:
        return f"Ошибка при получении погоды: {e}"
```


In [43]:
# TODO: Реализуйте Инструмент 1
from langchain_core.tools import tool

@tool
def compound_interest(
    principal: float,
    annual_rate: float,
    years: float,
    compounds_per_year: int
) -> str:
    """
    Вычисляет итоговую сумму по формуле сложных процентов.

    Используй этот инструмент, когда пользователь просит рассчитать
    доход по вкладу, инвестициям, накоплениям или сложным процентам.

    Args:
        principal: Начальная сумма вклада в рублях.
        annual_rate: Годовая процентная ставка (в процентах).
        years: Срок вклада в годах.
        compounds_per_year: Количество капитализаций процентов в год
            (например: 1 — ежегодно, 12 — ежемесячно, 365 — ежедневно).

    Returns:
        Строка с итоговой суммой и начисленными процентами.
    """

    try:
        rate = annual_rate / 100

        amount = principal * (
            1 + rate / compounds_per_year
        ) ** (compounds_per_year * years)

        interest = amount - principal

        return (
            f"Начальная сумма: {principal:.2f} ₽\n"
            f"Итоговая сумма: {amount:.2f} ₽\n"
            f"Начисленные проценты: {interest:.2f} ₽"
        )

    except Exception as e:
        return f"Ошибка при расчете сложных процентов: {e}"


In [44]:
# TODO: Реализуйте Инструмент 2
from langchain_core.tools import tool
import requests
from urllib.parse import quote


@tool
def get_weather(city: str) -> str:
    """
    Возвращает текущую погоду в указанном городе.

    Используй этот инструмент, когда пользователь спрашивает
    о текущей погоде, температуре, ветре или погодных условиях.

    Args:
        city: Название города.

    Returns:
        Информация о текущей погоде.
    """
    try:
        url = f"https://wttr.in/{quote(city)}?format=j1"

        response = requests.get(
            url,
            timeout=10,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        response.raise_for_status()

        data = response.json()

        current = data["current_condition"][0]

        return (
            f"Погода в городе {city}:\n"
            f"Температура: {current['temp_C']}°C\n"
            f"Ощущается как: {current['FeelsLikeC']}°C\n"
            f"Описание: {current['weatherDesc'][0]['value']}\n"
            f"Влажность: {current['humidity']}%\n"
            f"Ветер: {current['windspeedKmph']} км/ч"
        )

    except Exception as e:
        return f"Ошибка получения погоды: {str(e)}"

In [45]:
# TODO: Реализуйте Инструмент 3

import requests
import xml.etree.ElementTree as ET
from langchain_core.tools import tool


@tool
def get_currency_rate(currency: str) -> str:
    """
    Возвращает текущий официальный курс иностранной валюты
    по отношению к российскому рублю.

    Используй этот инструмент, когда пользователь спрашивает
    текущий курс валют (например USD, EUR, CNY, KZT и т.д.).

    Args:
        currency: Код валюты в формате ISO (например: USD, EUR, CNY, GBP, KZT).

    Returns:
        Строка с текущим курсом валюты к рублю.
    """

    try:
        response = requests.get(
            "https://www.cbr.ru/scripts/XML_daily.asp",
            timeout=10
        )
        response.raise_for_status()

        root = ET.fromstring(response.content)

        currency = currency.upper()

        for valute in root.findall("Valute"):

            char_code = valute.find("CharCode").text

            if char_code == currency:

                name = valute.find("Name").text
                nominal = valute.find("Nominal").text
                value = valute.find("Value").text

                return (
                    f"Курс {name} ({currency}):\n"
                    f"{nominal} {currency} = {value} RUB"
                )

        return f"Валюта с кодом '{currency}' не найдена."

    except Exception as e:
        return f"Ошибка при получении курса валют: {e}"


### 1.4 Промпт-инженерия и создание ReAct агента (10 баллов)

**Задание:**
1. Напишите системный промпт для агента. Задайте ему персону (например, "опытный финансовый консультант" или "строгий корпоративный помощник").
2. Промпт должен явно запрещать агенту отвечать на вопросы, выходящие за рамки его инструментов - это защита от галлюцинаций.
3. Создайте ReAct агента с помощью LangChain и подключите к нему ваши инструменты.

**Подсказки:**
- В LangChain используйте `create_react_agent` из `langchain.agents` и `AgentExecutor`.
- Передайте системный промпт через `ChatPromptTemplate` или параметр `agent_kwargs`.
- Установите `verbose=True` в `AgentExecutor` - так вы будете видеть все промежуточные шаги (мысли агента, вызовы инструментов, ответы инструментов). Это очень полезно для отладки.
- Установите `handle_parsing_errors=True` - это защитит от падений при некорректном ответе LLM.
- Параметр `max_iterations` ограничивает количество шагов агента и защищает от бесконечных циклов.

Пример создания агента:
```python
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="anthropic/claude-3.5-sonnet", ...)
tools = [tool_1, tool_2, tool_3]

# Можно взять готовый промпт из hub или написать свой
prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)
```


In [46]:
import os

from google.colab import userdata
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


# Загрузка ключа OpenRouter из Google Colab Secrets

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")


# TODO: Напишите системный промпт с персоной и запретом на ответы вне компетенции

system_prompt = """
Вы — строгий корпоративный помощник.

Вы помогаете пользователям только с помощью следующих инструментов:

1. Расчет сложных процентов.
2. Получение текущей погоды через wttr.in.
3. Получение текущего курса валют Центрального банка РФ.

Строгие правила:

- Используйте инструменты только тогда, когда запрос пользователя требует их применения.
- Отвечайте только на вопросы, которые можно решить с помощью доступных инструментов.
- Если вопрос выходит за рамки ваших инструментов, вежливо сообщите об этом пользователю.
- Никогда не придумывайте данные.
- Не используйте внешние знания вместо результатов инструментов.
- Всегда основывайте ответы на информации, полученной от инструментов.
"""


# TODO: Инициализируйте LLM

llm = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0
)


# TODO: Соберите список инструментов

tools = [
    compound_interest,
    get_weather,
    get_currency_rate
]


# TODO: Создайте ReAct агента

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

/tmp/ipykernel_5189/260052622.py:56: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


### 1.5 Тестирование базового агента (10 баллов)

Протестируйте вашего агента на различных запросах. Покажите вывод промежуточных шагов.

Задайте минимум 3 запроса:
1. Запрос, требующий вызова только одного инструмента.
2. Сложный запрос, требующий вызова двух инструментов последовательно.
3. Провокационный запрос вне компетенции агента (проверка защиты от галлюцинаций).

**Подсказка:** Используйте `agent_executor.invoke({"input": "ваш запрос"})`. Вывод `verbose=True` покажет все шаги рассуждений.


In [52]:
# проверка инстурмента
result = agent_executor.invoke(
    {
        "messages": [
            (
                "user",
                "Какая погода в Москве сейчас?"
            )
        ]
    }
)

print(result["messages"][-1].content)

Поскольку вы попросили о погоде в Москве, я ответил на ваш вопрос, используя инструмент get_weather.


In [48]:
# проверка двух инструментов последвоательно
result = agent_executor.invoke(
    {
        "messages": [
            (
                "user",
                "Какая сейчас погода в Москве и сколько будет стоить 100 долларов в рублях?"
            )
        ]
    }
)

print(result["messages"][-1].content)

Стоимость 100 долларов в рублях составляет 7803,08 рублей.


In [49]:
# проверка на галлюцинации
result = agent_executor.invoke(
    {
        "messages": [
            (
                "user",
                "Напиши мне прогноз курса биткоина через год"
            )
        ]
    }
)

print(result["messages"][-1].content)

К сожалению, я не могу предоставить вам прогноз курса биткоина или любой другой криптовалюты. Мои возможности ограничены доступными инструментами, и я не могу предсказывать будущие события или рыночные тенденции.

Если вы хотите получить информацию о текущем курсе биткоина, я могу предложить вам использовать инструмент "get_currency_rate" для получения текущего курса биткоина к рублю. Однако, я не могу предсказывать, как будет меняться курс через год.

Если вы хотите получить информацию о рыночных тенденциях или прогнозах, я рекомендую вам обратиться к профессиональным аналитикам или экспертам в области криптовалют.


In [50]:
# проверка на вопрос вне компетенции
result = agent_executor.invoke(
    {
        "messages": [
            (
                "user",
                "Кто победит на президентских выборах в России в 2030 году?"
            )
        ]
    }
)

print(result["messages"][-1].content)

Извините, но я не могу предоставить информацию о результатах будущих событий, включая президентские выборы в России в 2030 году. Мои возможности ограничены доступными инструментами и данными, и я не могу предсказывать будущее или предоставлять информацию, которая не подтверждена данными. Если вы хотите узнать о результатах прошлых президентских выборов в России или о текущих политических событиях, я могу попытаться помочь вам с этим.


### 1.6 Анализ рисков и идеи по улучшению (10 баллов)

Это задание не оценивается в баллах, но является обязательным. Здесь вы должны проявить критическое мышление и осмыслить то, что построили.

**Задание:** Напишите развернутый анализ (минимум 300 слов) в ячейке ниже, ответив на следующие вопросы:

**Риски текущей реализации:**
- Какие ошибки может совершить ваш агент? Приведите конкретные примеры запросов, на которых он может сломаться или дать неверный ответ.
- Что произойдет, если один из внешних API (погода, курсы) будет недоступен? Как агент обработает эту ситуацию?
- Насколько надежен ваш системный промпт? Можно ли обойти его ограничения с помощью хитро сформулированного запроса (prompt injection)?
- Какие риски несет использование больших LLM через внешние API (задержки, стоимость, утечка данных)?

**Гипотезы по улучшению:**
- Как можно улучшить качество поиска в инструменте базы знаний? Что если заменить keyword-поиск на семантический (с эмбеддингами)?
- Как можно сделать агента более устойчивым к ошибкам инструментов? Например, добавить логику повторных попыток или fallback-инструменты.
- Что изменится, если заменить большую LLM на маленькую локальную модель? Какие задачи пострадают в первую очередь?
- Как можно добавить память агенту, чтобы он помнил контекст предыдущих разговоров?

**Идеи по расширению:**
- Какие еще инструменты было бы полезно добавить для вашего конкретного сценария использования?
- Как бы вы оценивали качество работы агента в продакшене? Какие метрики использовали бы?


**Ваш анализ:**

# Анализ рисков и идеи по улучшению

# 1. Риски текущей реализации

### Какие ошибки может совершить агент?

Текущая реализация агента может допускать ошибки, такие как неправильная обработка результата инструмента. Например, при запросе текущей погоды агент может вызвать функцию `get_weather`, но вместо отображения полученных данных вернуть пользователю только сообщение о том, что инструмент был использован.


### Что произойдет при недоступности внешних API?

Работа инструментов зависит от внешних сервисов:

- API wttr.in для получения погоды;
- API Центрального банка России для получения курсов валют.

Если один из сервисов будет недоступен, инструмент не сможет получить данные. В текущей реализации предусмотрена обработка ошибок через `try/except`, поэтому программа не завершится с ошибкой, а вернет пользователю сообщение о проблеме.

### Насколько надежен системный промпт?

Системный промпт является важным элементом защиты агента. В нем задаются ограничения, например:

- отвечать только с использованием доступных инструментов;
- не придумывать данные;
- отказываться от запросов вне области компетенции.

Однако системный промпт не является абсолютной защитой. Пользователь может попытаться обойти ограничения

### Какие риски несет использование больших LLM через внешние API?

Использование больших языковых моделей через внешние API имеет несколько недостатков.

#### 1. Задержки

Каждый запрос пользователя отправляется во внешний сервис, поэтому увеличивается время ответа. При большом количестве пользователей задержки могут стать существенной проблемой.

#### 2. Стоимость

Большие языковые модели требуют оплаты за использование API. При высокой нагрузке стоимость обработки запросов может значительно увеличиться.

#### 3. Утечка данных

При использовании внешнего API пользовательские данные отправляются стороннему провайдеру. Для корпоративных приложений это может создавать проблемы с конфиденциальностью.

#### 4. Зависимость от внешнего сервиса

Если API модели станет недоступным, работа агента будет нарушена.

Для критичных систем может быть более подходящим вариантом использование локальных моделей или закрытой инфраструктуры.

---

#2. Гипотезы по улучшению

Если в агент добавить поиск по базе знаний, простой keyword-поиск можно заменить на семантический поиск с использованием эмбеддингов.

### Как сделать агента более устойчивым к ошибкам инструментов?

Для повышения надежности можно добавить следующие механизмы:

- повторные попытки вызова инструмента при временных ошибках;
- обработку разных типов исключений;
- резервные инструменты;
- проверку полученных результатов.

Например:

Если сервис погоды недоступен, агент может повторить запрос через несколько секунд или использовать альтернативный погодный API.

Также можно добавить проверку результата перед передачей пользователю, чтобы агент не отправлял пустые или некорректные данные.


### Что изменится при замене большой LLM на маленькую локальную модель?

Использование небольшой локальной модели даст преимущества:

- снижение стоимости;
- уменьшение задержек;
- возможность работы без внешнего API;
- больший контроль над данными.

Однако качество работы может снизиться.

В первую очередь пострадают:

- понимание сложных инструкций;
- способность выполнять многошаговые задачи;
- выбор правильных инструментов;
- обработка нестандартных запросов.

Большие модели лучше подходят для сложного планирования и работы с несколькими инструментами одновременно.


### Как добавить память агенту?

Чтобы агент помнил предыдущие сообщения пользователя, можно добавить механизм памяти.

Возможные варианты:

- сохранение истории диалогов в базе данных;
- использование LangGraph Memory.

---
# 3. Идеи по расширению

## Какие дополнительные инструменты можно добавить?

Для расширения возможностей агента можно добавить:

- поиск по внутренней базе документов;
- анализ PDF-файлов;
- работу с таблицами;
- отправку электронной почты;
- календарь и планирование задач;
- поиск актуальных новостей;
- генерацию отчетов;
- подключение корпоративных систем.

Для корпоративного помощника особенно полезным будет инструмент поиска по внутренней документации компании.

## Как оценивать качество работы агента в production?

Для оценки качества агента можно использовать следующие метрики:

*   Процент задач, которые агент успешно выполнил.
*   Насколько правильно агент выбирает необходимые инструменты.
*   Среднее время формирования ответа пользователю.
*   Количество ошибок внешних инструментов.
*   Стоимость обработки одного пользовательского запроса.
*   Оценка качества работы агентом пользователями.
*   Количество случаев, когда агент создает недостоверную информацию.

---

## Итоговый вывод

Созданный агент демонстрирует базовую способность использовать внешние инструменты и выполнять задачи через взаимодействие с языковой моделью. Однако текущая реализация имеет ограничения, связанные с надежностью API, возможными ошибками выбора инструментов и уязвимостью к prompt injection.

Для использования агента в реальных production-системах необходимо добавить более надежную обработку ошибок, семантический поиск, механизм памяти, дополнительные проверки безопасности и систему оценки качества работы.



---
# Часть 2. Продвинутая (50 баллов)

В этой части вы переведете агента на рельсы LangGraph, добавите разделение ролей (Оркестратор и субагенты) и внедрите механизм безопасности (Human-in-the-loop).

**Напоминание:** Вы можете использовать любые технологии. Описанный ниже подход через LangGraph - рекомендация, а не требование.


### 2.1 Переход на LangGraph (10 баллов)

Перепишите базового агента из Части 1 с использованием LangGraph.

**Что нужно сделать:**
1. Определить граф состояния (`StateGraph`) с узлами для LLM и для инструментов.
2. Настроить `conditional_edges` для маршрутизации: если LLM вызвал инструмент - идем в узел инструментов, иначе - завершаем.
3. Скомпилировать граф и визуализировать его.

**Подсказки:**
- Используйте `MessagesState` как базовое состояние - это удобная обертка над списком сообщений.
- Узел агента вызывает LLM с привязанными инструментами: `llm.bind_tools(tools)`.
- Для узла инструментов используйте готовый `ToolNode` из `langgraph.prebuilt`.
- Для маршрутизации используйте `tools_condition` из `langgraph.prebuilt` - он уже умеет определять, нужно ли вызывать инструменты.
- Для визуализации: `graph.get_graph().draw_mermaid_png()`.

Пример скелета графа:
```python
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
graph = builder.compile()
```


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import Image, display

# TODO: Определите функцию узла агента (вызов LLM с инструментами)

# TODO: Соберите граф с узлами и ребрами

# TODO: Скомпилируйте граф
# graph = builder.compile()

# TODO: Визуализируйте граф
# display(Image(graph.get_graph().draw_mermaid_png()))


### 2.2 - 2.3 Промпт для Оркестратора и создание субагентов (15 баллов)

**Задание:**
1. Разделите ваши 3 инструмента между двумя субагентами (например, Агент-Аналитик и Агент-Информатор).
2. Напишите системный промпт для Оркестратора, описывающий компетенции каждого субагента и правила маршрутизации.
3. Реализуйте субагентов как отдельные узлы в графе.
4. Оркестратор должен анализировать запрос пользователя и направлять его нужному субагенту.

**Подсказки по промпту Оркестратора:**
- Четко опишите, что умеет каждый субагент. Чем точнее описание - тем лучше маршрутизация.
- Укажите, что делать, если запрос не подходит ни одному субагенту.
- Попросите Оркестратора объяснять свое решение о маршрутизации.

**Подсказки по архитектуре:**
- Каждый субагент - это отдельная функция-узел в графе, которая вызывает своего LLM с набором инструментов.
- Оркестратор может быть реализован как узел с `conditional_edges`, которые смотрят на решение LLM.
- Для передачи контекста между агентами используйте поле `messages` в состоянии графа.
- Можно добавить кастомные поля в состояние (например, `current_agent: str`) для отслеживания маршрута.

Пример структуры мультиагентного графа:
```python
class AgentState(MessagesState):
    current_agent: str  # какой агент сейчас работает

def orchestrator_node(state):
    # LLM решает, кому делегировать
    ...

def analyst_agent_node(state):
    # Субагент с инструментами анализа
    ...

def info_agent_node(state):
    # Субагент с инструментами получения информации
    ...
```


In [ ]:
# TODO: Напишите системный промпт для Оркестратора
orchestrator_prompt = """
Вы - Оркестратор. Ваша задача - принять запрос пользователя и направить его нужному субагенту.

У вас есть два субагента:
1. Агент-Аналитик: умеет [опишите компетенции].
2. Агент-Информатор: умеет [опишите компетенции].

Правила маршрутизации:
- Если запрос требует [условие] - направьте к Агент-Аналитику.
- Если запрос требует [условие] - направьте к Агент-Информатору.
- Если запрос не подходит ни одному - вежливо откажитесь.
"""

# TODO: Определите узлы субагентов

# TODO: Определите узел Оркестратора и логику маршрутизации

# TODO: Соберите мультиагентный граф и визуализируйте его


### 2.4 Human-in-the-loop (Безопасность) (10 баллов)

**Задание:**
1. Добавьте инструмент `send_report_to_management` (может просто печатать текст или сохранять в файл).
2. Настройте граф так, чтобы перед вызовом этого инструмента выполнение приостанавливалось и ожидало ручного подтверждения.

**Подсказки:**
- В LangGraph для паузы используется параметр `interrupt_before=["tools"]` при компиляции графа.
- Для сохранения состояния во время паузы нужен `checkpointer`. Используйте `MemorySaver` для тестирования.
- Каждый запуск графа должен иметь уникальный `thread_id` в `config` - это идентификатор сессии.
- Чтобы возобновить выполнение, вызовите граф повторно с тем же `thread_id` и `None` в качестве входных данных.
- Используйте `graph.get_state(config)` чтобы проверить текущее состояние и убедиться, что граф на паузе.

Пример паузы и возобновления:
```python
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph = builder.compile(checkpointer=memory, interrupt_before=["tools"])

config = {"configurable": {"thread_id": "session-1"}}

# Первый запуск - граф остановится перед вызовом инструмента
result = graph.invoke({"messages": [("user", "запрос")]}, config)

# Проверяем состояние
state = graph.get_state(config)
print("Граф на паузе:", state.next)

# Возобновляем выполнение (подтверждение)
final_result = graph.invoke(None, config)
```


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# TODO: Создайте инструмент send_report_to_management
@tool
def send_report_to_management(report_text: str) -> str:
    """Отправляет финальный отчет руководству. Используй только когда пользователь явно просит отправить отчет.
    Args:
        report_text: Текст отчета для отправки.
    """
    # TODO: Ваша реализация (например, сохранить в файл или напечатать)
    pass

# TODO: Добавьте инструмент одному из субагентов

# TODO: Создайте checkpointer и скомпилируйте граф с interrupt_before
# memory = MemorySaver()
# graph_with_hitl = builder.compile(checkpointer=memory, interrupt_before=["tools"])


### 2.5 Финальное тестирование мультиагентной системы (5 баллов)

Продемонстрируйте полный цикл работы вашей мультиагентной системы.

Задайте сложный запрос, который:
1. Требует делегирования от Оркестратора к субагенту.
2. Заканчивается вызовом инструмента `send_report_to_management`.

Покажите все четыре этапа: запуск, пауза перед отправкой, ручное подтверждение, финальный ответ.

**Подсказка:** Выводите промежуточные состояния графа, чтобы было видно, как меняется `state.next` до и после подтверждения.


In [ ]:
# TODO: Этап 1 - Запустите граф с комплексным запросом
config = {"configurable": {"thread_id": "final-test-1"}}
# result = graph_with_hitl.invoke({"messages": [("user", "ваш запрос")]}, config)


In [ ]:
# TODO: Этап 2 - Проверьте, что граф на паузе
# state = graph_with_hitl.get_state(config)
# print("Следующий шаг:", state.next)
# print("Последнее сообщение:", state.values["messages"][-1])


In [ ]:
# TODO: Этап 3 - Дайте подтверждение и возобновите выполнение
# final_result = graph_with_hitl.invoke(None, config)


In [ ]:
# TODO: Этап 4 - Выведите финальный ответ
# print(final_result["messages"][-1].content)


### 2.6 Анализ рисков и идеи по улучшению мультиагентной системы (10 баллов)

Это задание не оценивается в баллах, но является обязательным. Здесь вы должны проявить системное мышление и осмыслить архитектуру, которую построили.

**Задание:** Напишите развернутый анализ (минимум 400 слов) в ячейке ниже, ответив на следующие вопросы:

**Риски мультиагентной архитектуры:**
- Что произойдет, если Оркестратор неправильно определит нужного субагента? Как часто это может происходить и почему?
- Как растет стоимость и задержка при добавлении новых субагентов? Когда мультиагентность становится избыточной?
- Насколько надежен механизм Human-in-the-loop? Что если человек нажмет "подтвердить" не глядя?
- Какие риски несет общее состояние (`messages`) между агентами? Может ли один субагент "запутать" другого?

**Гипотезы по улучшению:**
- Как можно улучшить качество маршрутизации Оркестратора? Например, добавить классификатор намерений (intent classifier) перед Оркестратором.
- Как добавить долгосрочную память агентам? Например, сохранять важные факты из разговоров в векторную базу данных.
- Как реализовать параллельное выполнение субагентов, если запрос требует работы нескольких из них одновременно?
- Как можно автоматически оценивать качество ответов агентов (LLM-as-a-judge)?

**Идеи по расширению:**
- Какие новые субагенты и инструменты сделали бы вашу систему значительно полезнее?
- Как бы вы развернули эту систему в продакшене? Какую инфраструктуру выбрали бы?
- Как реализовать мониторинг и трассировку работы агентов в реальном времени (например, через Arize Phoenix или LangSmith)?
- Как обеспечить безопасность системы от prompt injection атак, когда злоумышленник пытается через пользовательский запрос изменить поведение агента?


**Ваш анализ:**

...

---
**Поздравляем с завершением домашнего задания!**

Вы прошли путь от базового ReAct агента до мультиагентной системы с защитой и человеком в контуре. Это фундамент для построения реальных продакшен-систем на базе LLM.